In [1]:
import csv
import os
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import combinations
from typing import Dict, List, Set, Tuple

from utils import (
    BASE,
    PATHWAYS_FILE,
    PW_PAIR,
    ROUTES_FILE,
    STOPS_FILE,
    STOP_TIMES_FILE,
    TRANSFERS_FILE,
    TRIPS_FILE,
    build_graph_and_coverage,
    check_missing_files,
    check_trip,
    iter_pathway_pairs,
    load_from_stop_ids,
    load_pathway_ids,
    load_platform_pairs_present,
    load_platforms_by_name,
    load_route_ids,
    load_stop_ids,
    load_stop_names,
    load_stops_info,
    load_to_stop_ids,
    load_transfer_pairs,
    load_trip_ids,
    make_signature,
    read_dict_rows,
    sniff_dialect,
)

### File connection checks

#### All stop_id from pathways and stop_times exist in stops

In [2]:
def main():
    check_missing_files([PATHWAYS_FILE, STOP_TIMES_FILE, STOPS_FILE])

    pathways_from_stop_ids = load_from_stop_ids(PATHWAYS_FILE)
    pathways_to_stop_ids = load_to_stop_ids(PATHWAYS_FILE)
    stop_times_stop_ids = load_stop_ids(STOP_TIMES_FILE)
    stops_stop_ids = load_stop_ids(STOPS_FILE)

    missing_pathways_from = sorted(ids for ids in pathways_from_stop_ids if ids not in stops_stop_ids)
    missing_pathways_to = sorted(ids for ids in pathways_to_stop_ids if ids not in stops_stop_ids)
    missing_stop_times = sorted(ids for ids in stop_times_stop_ids if ids not in stops_stop_ids)

    if not missing_pathways_from:
        print(" - All correct: no from_stop_id from pathways.txt is missing in stops.txt")
    else:
        print(f" - MISSING {len(missing_pathways_from)} from_stop_id from pathways.txt in stops.txt:")
        for sid in missing_pathways_from:
            print("   -", sid)
    if not missing_pathways_to:
        print(" - All correct: no to_stop_id from pathways.txt is missing in stops.txt")
    else:        
        print(f" - MISSING {len(missing_pathways_to)} to_stop_id from pathways.txt in stops.txt:")
        for sid in missing_pathways_to:
            print("   -", sid)
    if not missing_stop_times:
        print(" - All correct: no stop_id from stop_times.txt is missing in stops.txt")
    else:
        print(f" - MISSING {len(missing_stop_times)} stop_id from stop_times.txt in stops.txt:")
        for sid in missing_stop_times:
            print("   -", sid)

main()

 - All correct: no from_stop_id from pathways.txt is missing in stops.txt
 - All correct: no to_stop_id from pathways.txt is missing in stops.txt
 - All correct: no stop_id from stop_times.txt is missing in stops.txt


#### All route_id from trips exist in routes

In [3]:
def main():
    check_missing_files([TRIPS_FILE, ROUTES_FILE])

    trips_route_ids = load_route_ids(TRIPS_FILE)
    routes_route_ids = load_route_ids(ROUTES_FILE)

    missing = sorted(ids for ids in trips_route_ids if ids not in routes_route_ids)

    print(f"Unique route_id in trips: {len(trips_route_ids)}")
    print(f"Unique route_id in routes: {len(routes_route_ids)}")

    if not missing:
        print("All correct: all route_id present in trips also appear in routes.")
    else:
        print(f"MISSING {len(missing)} route_id (present in trips but not in routes):")
        for rid in missing:
            print(f"- {rid}")

main()

Unique route_id in trips: 114
Unique route_id in routes: 114
All correct: all route_id present in trips also appear in routes.


#### All trip_id from stop_times exist in trips

In [4]:
def main():
    check_missing_files([STOP_TIMES_FILE, TRIPS_FILE])

    stop_times_trip_ids = load_trip_ids(STOP_TIMES_FILE)
    trips_trip_ids = load_trip_ids(TRIPS_FILE)

    missing= sorted(ids for ids in stop_times_trip_ids if ids not in trips_trip_ids)    

    print(f"Unique trip_id in stop_times: {len(stop_times_trip_ids)}")
    print(f"Unique trip_id in trips: {len(trips_trip_ids)}")

    if not missing:
        print("All correct: all trip_id present in stop_times also appear in trips.")    
    else:
        print(f"MISSING {len(missing)} trip_id (present in stop_times but not in trips):")
        for tid in missing:
            print(f"- {tid}")

main()

Unique trip_id in stop_times: 48987
Unique trip_id in trips: 48987
All correct: all trip_id present in stop_times also appear in trips.


### Pathways checks

#### Check that transfers are within pathways

Check that each (from_stop_id, to_stop_id) from transfers has a pathway_id = "PW.{from_stop_id}_{to_stop_id}" in pathways


In [5]:
def main() -> None:
    check_missing_files([PATHWAYS_FILE, TRANSFERS_FILE])

    pathways_ids = load_pathway_ids(PATHWAYS_FILE)
    transfers_pairs = list(load_transfer_pairs(TRANSFERS_FILE))

    expected_ids = [f"PW.{a}_{b}" for a, b in transfers_pairs]
    missing = [(a, b, eid) for (a, b), eid in zip(transfers_pairs, expected_ids) if eid not in pathways_ids]

    print(f"Total transfers: {len(transfers_pairs)}")
    print(f"Total pathways: {len(pathways_ids)}")

    if not missing:
        print("All correct: all rows in transfers have a corresponding pathway (format 'PW.{from_stop_id}_{to_stop_id}').")
    else:
        print(f"MISSING {len(missing)} pathways for specific transfers (showing all):")
        for a, b, eid in missing:
            print(f"- from_stop_id={a!r}, to_stop_id={b!r} -> expected pathway_id={eid!r}")


main()

Total transfers: 60
Total pathways: 1063
All correct: all rows in transfers have a corresponding pathway (format 'PW.{from_stop_id}_{to_stop_id}').


#### Check that pathways have both directions

Check symmetry in pathways: for each PW.x_y check that PW.y_x exists.
When the inverse is missing, show the stop_name and coordinates (lat/lon) for each a and b.


In [6]:
def main() -> None:
    check_missing_files([PATHWAYS_FILE, STOPS_FILE])
    
    pids = load_pathway_ids(PATHWAYS_FILE)

    # Load stop info: name and coordinates
    stops_info = load_stops_info(STOPS_FILE)

    # Consider only those following the format 'PW.a_b'
    candidate_pids = {pid for pid in pids if PW_PAIR.match(pid) is not None}

    missing: List[Tuple[str, str, str, str]] = []
    for pid in sorted(candidate_pids):
        m = PW_PAIR.match(pid)
        a, b = m.group('a'), m.group('b')
        reverse_id = f"PW.{b}_{a}"
        if reverse_id not in pids:
            missing.append((pid, reverse_id, a, b))

    print(f"Total pathways: {len(pids)}")
    print(f"Candidates with format 'PW.a_b': {len(candidate_pids)}")

    if not missing:
        print("All correct: for each pathway 'PW.x_y' there also exists 'PW.y_x'.")
    else:
        print(f"MISSING {len(missing)} inverse pathways (showing all):")
        for pid, rid, a, b in missing:
            print(f"- Exists {pid!r} but missing its inverse {rid!r}")
            a_name, a_lat, a_lon = stops_info.get(a, ("(no name)", "", ""))
            b_name, b_lat, b_lon = stops_info.get(b, ("(no name)", "", ""))
            print(f"  · {a} — {a_name} (lat={a_lat}, lon={a_lon})")
            print(f"  · {b} — {b_name} (lat={b_lat}, lon={b_lon})")

main()

Total pathways: 1063
Candidates with format 'PW.a_b': 1063
MISSING 1 inverse pathways (showing all):
- Exists 'PW.1.120_E.12001' but missing its inverse 'PW.E.12001_1.120'
  · 1.120 — Plaça de Sants (lat=41.375353, lon=2.138154)
  · E.12001 — Alcolea (escala mecànica) (lat=41.375457, lon=2.137077)


#### Check that traversal_time is the same in both directions

In [7]:
def main() -> None:
    check_missing_files([PATHWAYS_FILE])

    # Reuse shared parser to keep only valid pathway_id format 'PW.a_b'.
    valid_pw_ids: Set[str] = {pid for pid, _, _ in iter_pathway_pairs(PATHWAYS_FILE)}

    # Build map: pathway_id -> traversal_time (string as stored in file).
    traversal_by_pid: Dict[str, str] = {}
    total_rows = 0
    for r in read_dict_rows(PATHWAYS_FILE):
        total_rows += 1
        pid = r.get('pathway_id', '').strip()
        if pid in valid_pw_ids:
            traversal_by_pid[pid] = r.get('traversal_time', '').strip()

    compared_pairs: Set[Tuple[str, str]] = set()
    mismatches: List[Tuple[str, str, str, str]] = []
    missing_reverse = 0

    for pid, a, b in iter_pathway_pairs(PATHWAYS_FILE):
        reverse_id = f"PW.{b}_{a}"

        # Compare each undirected pair only once.
        pair_key = tuple(sorted((pid, reverse_id)))
        if pair_key in compared_pairs:
            continue
        compared_pairs.add(pair_key)

        if reverse_id not in traversal_by_pid:
            missing_reverse += 1
            continue

        t_ab = traversal_by_pid.get(pid, '')
        t_ba = traversal_by_pid[reverse_id]
        if t_ab != t_ba:
            mismatches.append((pid, t_ab, reverse_id, t_ba))

    print(f"Total rows in pathways: {total_rows}")
    print(f"Rows with pathway_id format 'PW.a_b': {len(valid_pw_ids)}")
    print(f"Pairs compared (with reverse present): {len(compared_pairs) - missing_reverse}")

    if missing_reverse:
        print(f"Skipped {missing_reverse} pairs because reverse pathway is missing.")

    if not mismatches:
        print("All correct: traversal_time matches between 'PW.a_b' and 'PW.b_a' for all comparable pairs.")
    else:
        print(f"MISSING equal traversal_time in {len(mismatches)} pathway pairs:")
        for pid, t_ab, reverse_id, t_ba in sorted(mismatches):
            print(f"- {pid}: traversal_time={t_ab!r} | {reverse_id}: traversal_time={t_ba!r}")

main()

Total rows in pathways: 1063
Rows with pathway_id format 'PW.a_b': 1063
Pairs compared (with reverse present): 531
Skipped 1 pairs because reverse pathway is missing.
All correct: traversal_time matches between 'PW.a_b' and 'PW.b_a' for all comparable pairs.


#### Check if traversal_time is multiple of 15

In [8]:
def main() -> None:
    check_missing_files([PATHWAYS_FILE])

    total_rows = 0
    valid_pw_rows = 0
    missing_traversal: List[str] = []
    non_numeric: List[Tuple[str, str]] = []
    not_multiple_15: List[Tuple[str, int]] = []

    for r in read_dict_rows(PATHWAYS_FILE):
        total_rows += 1
        pid = r.get('pathway_id', '').strip()
        if not pid or PW_PAIR.match(pid) is None:
            continue
        valid_pw_rows += 1

        t_raw = r.get('traversal_time', '').strip()
        if not t_raw:
            missing_traversal.append(pid)
            continue

        try:
            t_val = int(t_raw)
        except Exception:
            non_numeric.append((pid, t_raw))
            continue

        if t_val % 15 != 0:
            not_multiple_15.append((pid, t_val))

    print(f"Total rows in pathways: {total_rows}")
    print(f"Rows with pathway_id format 'PW.a_b': {valid_pw_rows}")

    if not missing_traversal and not non_numeric and not not_multiple_15:
        print("All correct: traversal_time is present, numeric, and multiple of 15 for all PW pathways.")
        return

    if missing_traversal:
        print(f"MISSING traversal_time in {len(missing_traversal)} pathways:")
        for pid in sorted(missing_traversal):
            print(f"- {pid}")

    if non_numeric:
        print(f"NON-NUMERIC traversal_time in {len(non_numeric)} pathways:")
        for pid, raw in sorted(non_numeric):
            print(f"- {pid}: traversal_time={raw!r}")

    if not_multiple_15:
        print(f"NOT multiple of 15 in {len(not_multiple_15)} pathways:")
        for pid, t_val in sorted(not_multiple_15):
            print(f"- {pid}: traversal_time={t_val}")

main()

Total rows in pathways: 1063
Rows with pathway_id format 'PW.a_b': 1063
All correct: traversal_time is present, numeric, and multiple of 15 for all PW pathways.


#### Check that each entrance is connected to a platform

Check that each stop_id starting with 'E.' has at least one pathway with a platform '1.'
Requirement: pathway_id of the type 'PW.E.xxx_1.yyy' or 'PW.1.yyy_E.xxx'


In [9]:
def main():
    check_missing_files([PATHWAYS_FILE, STOPS_FILE])

    e_stops_info: Dict[str, Tuple[str, str, str]] = {}
    for r in read_dict_rows(STOPS_FILE):
        sid = r.get('stop_id', '').strip()
        if not sid or not sid.startswith('E.'):
            continue
        name = r.get('stop_name', '').strip()
        lat = r.get('stop_lat', '').strip()
        lon = r.get('stop_lon', '').strip()
        e_stops_info[sid] = (name, lat, lon)

    pids = load_pathway_ids(PATHWAYS_FILE)

    # E.* with at least one PW connection to some 1.* (in any order)
    covered: Set[str] = set()
    for pid in pids:
        m = PW_PAIR.match(pid)
        if not m:
            continue
        a, b = m.group('a'), m.group('b')
        if a.startswith('E.') and b.startswith('1.'):
            covered.add(a)
        elif b.startswith('E.') and a.startswith('1.'):
            covered.add(b)

    missing = [e for e in e_stops_info.keys() if e not in covered]

    print(f"Entrances (E.*): {len(e_stops_info)}")
    print(f"Entrances with at least one pathway to a platform (1.*): {len(covered)}")

    if not missing:
        print("All correct: each E.* has at least one pathway 'PW.E.xxx_1.yyy' or 'PW.1.yyy_E.xxx'.")
    else:
        print(f"MISSING {len(missing)} E.* without any pathway to 1.* (showing all):")
        for e in missing:
            name, lat, lon = e_stops_info.get(e, ("(no name)", "", ""))
            print(f"- {e} — {name} (lat={lat}, lon={lon})")

main()

Entrances (E.*): 502
Entrances with at least one pathway to a platform (1.*): 502
All correct: each E.* has at least one pathway 'PW.E.xxx_1.yyy' or 'PW.1.yyy_E.xxx'.


#### Check that each platform is connected to an entrance

Check that each stop_id starting with '1.' (platform) has at least one pathway with an entrance 'E.'
That is, check if there exists some 'PW.1.xxx_E.yyy' or 'PW.E.yyy_1.xxx' for each '1.xxx'.
When one is missing, also show the associated stop_name and suggest a neighboring platform (1.*) that does have an entrance, with its entrance (E.*) and names.


In [10]:
def main():
    check_missing_files([PATHWAYS_FILE, STOPS_FILE])

    stop_ids = load_stop_ids(STOPS_FILE)
    stop_names = load_stop_names(STOPS_FILE)
    one_stops = sorted({s for s in stop_ids if s.startswith('1.')})

    pids = load_pathway_ids(PATHWAYS_FILE)

    # Platform graph and entrance coverage
    adj_1, one_to_entries, covered = build_graph_and_coverage(pids)

    missing = [s for s in one_stops if s not in covered]

    print(f"Total stops: {len(stop_ids)}")
    print(f"Platforms (1.*): {len(one_stops)}")
    print(f"Platforms with at least one pathway to an entrance (E.*): {len(covered)}")

    if not missing:
        print("All correct: each 1.* has at least one pathway 'PW.1.xxx_E.yyy' or 'PW.E.yyy_1.xxx'.")
        return

    print(f"MISSING {len(missing)} 1.* without any pathway to E.* (showing all):")
    for s in missing:
        nom = stop_names.get(s, "(no name)")
        print(f"- {s} — {nom}")
        # Suggest neighboring platforms that do have an entrance
        neigh = sorted(adj_1.get(s, []))
        sugg = [n for n in neigh if n in one_to_entries and one_to_entries[n]]
        if sugg:
            print("    Connected to platforms that do have at least one entrance:")
            for n in sugg:
                n_nom = stop_names.get(n, "(no name)")
                # Pick one entrance (or all)
                entries = sorted(one_to_entries[n])
                # Show one (the first) for simplicity
                e = entries[0]
                e_nom = stop_names.get(e, "(no name)")
                print(f"    · {n} — {n_nom} -> entrance {e} — {e_nom}")
        else:
            print("    (Not connected to any platform 1.* that has an entrance E.*)")

main()

Total stops: 3443
Platforms (1.*): 165
Platforms with at least one pathway to an entrance (E.*): 158
MISSING 7 1.* without any pathway to E.* (showing all):
- 1.1136 — Trinitat Nova
    Connected to platforms that do have at least one entrance:
    · 1.339 — Trinitat Nova -> entrance E.33901 — Palamós
    · 1.434 — Trinitat Nova -> entrance E.43401 — Pedrosa
- 1.221 — La Pau
    Connected to platforms that do have at least one entrance:
    · 1.413 — La Pau -> entrance E.1041301 — Ascensor - Rambla Guipúscoa / Ca n'Oliva
- 1.323 — Paral·lel
    Connected to platforms that do have at least one entrance:
    · 1.210 — Paral·lel -> entrance E.1021002 — Ascensor - Avinguda Paral·lel
- 1.915 — Torrassa
    Connected to platforms that do have at least one entrance:
    · 1.117 — Torrassa -> entrance E.1011711 — Ascensor - Av. Catalunya
- 1.930 — La Sagrera
    Connected to platforms that do have at least one entrance:
    · 1.133 — La Sagrera -> entrance E.1013301 — Ascensor - Meridiana | Ho

#### Check that there is a pathway between platforms of the same stop

Check that all platforms (1.*) with the same stop_name have pathways between each pair. Example: if 1.339, 1.434 and 1.1136 exist with the same name, pathways are needed between each pair (we only check one direction per pair; reciprocity is already validated in a previous cell).

In [11]:
def main():
    check_missing_files([PATHWAYS_FILE, STOPS_FILE])

    name_to_ones = load_platforms_by_name(STOPS_FILE)
    platform_pairs = load_platform_pairs_present(PATHWAYS_FILE)

    # Basic statistics
    total_ones = sum(len(v) for v in name_to_ones.values())
    groups_multi = {name: ids for name, ids in name_to_ones.items() if len(ids) >= 2}

    print(f"Total platforms (1.*): {total_ones}")
    print(f"Stops with multiple platforms (same name): {len(groups_multi)}")

    # Check all combinations per group
    missing: List[Tuple[str, str, str]] = []  # (name, u, v)
    for name, ids in sorted(groups_multi.items()):
        for u, v in combinations(sorted(ids), 2):
            if (u, v) not in platform_pairs:
                missing.append((name, u, v))

    if not missing:
        print("All correct: for each stop with multiple platforms, there are pathways between all platform pairs.")
    else:
        print(f"MISSING pathways between {len(missing)} platform pairs within the same stop (showing all):")
        for name, u, v in missing:
            print(f"- {name}: missing pathway between {u} and {v}")

main()

Total platforms (1.*): 165
Stops with multiple platforms (same name): 22
All correct: for each stop with multiple platforms, there are pathways between all platform pairs.


### stop_times checks

#### Duplicate full trip_id in stop_times

In [12]:
def main():
    check_missing_files([STOP_TIMES_FILE, TRIPS_FILE])

    # trip_id of interest (filtered by prefix '1.')
    trip_ids = load_trip_ids(TRIPS_FILE)
    trip_ids = sorted(ids for ids in trip_ids if ids.startswith("1."))
    trip_id_set = set(trip_ids)

    print(f"Unique trip_id in trips (prefix '1.'): {len(trip_ids)}")

    # Aggregate all stop_times rows per trip_id
    # Store tuples (stop_sequence_int, arrival_time, departure_time, stop_id)
    trips_rows: Dict[str, List[Tuple[int, str, str, str]]] = defaultdict(list)
    total_rows = 0
    for r in read_dict_rows(STOP_TIMES_FILE):
        total_rows += 1
        tid = r.get('trip_id', '')
        if tid not in trip_id_set:
            continue
        seq_s = r.get('stop_sequence', '')
        arr = r.get('arrival_time', '')
        dep = r.get('departure_time', '')
        sid = r.get('stop_id', '')
        try:
            seq = int(seq_s)
        except Exception:
            seq = 10**9
        trips_rows[tid].append((seq, arr, dep, sid))

    print(f"Total rows read from stop_times: {total_rows}")
    print(f"trip_id with at least one row in stop_times: {len(trips_rows)}")

    # Build signatures in parallel for each trip_id
    sig_to_trips: Dict[Tuple[Tuple[int, str, str, str], ...], List[str]] = defaultdict(list)
    max_workers = min(8, (os.cpu_count() or 4))
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(make_signature, item): item[0] for item in trips_rows.items()}
        for fut in as_completed(futures):
            tid, sig = fut.result()
            sig_to_trips[sig].append(tid)

    # Duplicate groups: signatures shared by >= 2 different trip_id
    duplicate_groups = [(sig, sorted(tids)) for sig, tids in sig_to_trips.items() if len(tids) >= 2]
    duplicate_groups.sort(key=lambda x: (len(x[1]), x[1]))

    if not duplicate_groups:
        print("No pair/group of trip_id with identical sequence and schedules was found.")
        return

    total_dup_trips = sum(len(tids) for _, tids in duplicate_groups)
    print(f"\nFound {len(duplicate_groups)} groups of trip_id with identical content ({total_dup_trips} trip_id in total).")

    len_counts = {}
    for _, tids in duplicate_groups:
        count = len(tids)
        len_counts[count] = len_counts.get(count, 0) + 1
    for count in sorted(len_counts.keys()):
        print(f"Groups with {count} trip_id: {len_counts[count]}")
        for sig, tids in duplicate_groups:
            if len(tids) == count:
                print(f"  Example group with {count} trip_id: {tids[:5]}{'...' if len(tids) > 5 else ''}")
                break

    # ---- Generate file with trip_id to eliminate ----
    # For each group, keep the first trip_id (alphabetically) and eliminate the rest
    to_eliminate: List[str] = []
    for _, tids in duplicate_groups:
        to_eliminate.extend(tids[1:])  # tids is already sorted

    out_path = os.path.join(BASE, "trip_ids_to_eliminate.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        for tid in sorted(to_eliminate):
            f.write(tid + "\n")

    kept = total_dup_trips - len(to_eliminate)
    print(f"\nTrip_id kept from groups (1 per group): {kept}")
    print(f"Trip_id to eliminate: {len(to_eliminate)}")
    print(f"Total valid trip_id after removing duplicates: {len(trip_ids) - len(to_eliminate)}")
    print(f"Generated file: {out_path}")

    # ---- Group details ----
    # for idx, (sig, tids) in enumerate(duplicate_groups, 1):
    #     print(f"\nGroup {idx}: {len(tids)} trip_id  (kept: {tids[0]})")
    #     for t in tids[1:]:
    #         print(f"  x {t}")
    #     print("Common content (sorted by stop_sequence):")
    #     limit = 100
    #     for i, (seq, arr, dep, sid) in enumerate(sig[:limit], 1):
    #         arr_show = arr if arr else "(none)"
    #         dep_show = dep if dep else "(none)"
    #         sid_show = sid if sid else "(none)"
    #         print(f"  {i:02d}. seq={seq}, arrival={arr_show}, departure={dep_show}, stop_id={sid_show}")
    #     if len(sig) > limit:
    #         print(f"  ... ({len(sig) - limit} more lines)")

main()

Unique trip_id in trips (prefix '1.'): 15088
Total rows read from stop_times: 1138586
trip_id with at least one row in stop_times: 15088

Found 4085 groups of trip_id with identical content (8196 trip_id in total).
Groups with 2 trip_id: 4065
  Example group with 2 trip_id: ['1.1.11654310', '1.1.11655306']
Groups with 3 trip_id: 14
  Example group with 3 trip_id: ['1.101.11427287', '1.101.11427796', '1.101.11428435']
Groups with 4 trip_id: 6
  Example group with 4 trip_id: ['1.104.11428897', '1.104.11430235', '1.104.11571861', '1.104.11572496']

Trip_id kept from groups (1 per group): 4085
Trip_id to eliminate: 4111
Total valid trip_id after removing duplicates: 10977
Generated file: c:\Users\Sarad\Escritorio\Mates\Cursos\Curs 2025-2026 (3r + 4t)\TFG\TFG\.src\gtfs\data\trip_ids_to_eliminate.txt


In [13]:
# Read trip_ids_to_eliminate.txt and create stop_times_cleaned.txt
# keeping only rows whose trip_id is NOT in the elimination list.

def main():
    eliminate_path = os.path.join(BASE, "trip_ids_to_eliminate.txt")
    check_missing_files([STOP_TIMES_FILE, eliminate_path])

    # Load trip_id to eliminate
    with open(eliminate_path, 'r', encoding='utf-8') as f:
        to_eliminate: Set[str] = {line.strip() for line in f if line.strip()}

    print(f"Trip_id to eliminate: {len(to_eliminate)}")

    # Read stop_times and write cleaned version
    out_path = os.path.join(BASE, "stop_times_cleaned.txt")
    dialect = sniff_dialect(STOP_TIMES_FILE)

    total_rows = 0
    kept_rows = 0
    removed_rows = 0

    with open(STOP_TIMES_FILE, 'r', encoding='utf-8-sig', newline='') as fin, \
         open(out_path, 'w', encoding='utf-8', newline='') as fout:

        reader = csv.DictReader(fin, dialect=dialect)
        if reader.fieldnames is None:
            raise RuntimeError("stop_times.txt has no header.")

        writer = csv.DictWriter(fout, fieldnames=reader.fieldnames, dialect=dialect)
        writer.writeheader()

        for row in reader:
            total_rows += 1
            tid = row.get('trip_id', '').strip()
            if tid in to_eliminate:
                removed_rows += 1
            else:
                writer.writerow(row)
                kept_rows += 1

    print(f"Total rows in stop_times.txt: {total_rows}")
    print(f"Rows removed: {removed_rows}")
    print(f"Rows kept: {kept_rows}")
    print(f"Cleaned file saved to: {out_path}")

main()

Trip_id to eliminate: 4111
Total rows in stop_times.txt: 1138586
Rows removed: 78215
Rows kept: 1060371
Cleaned file saved to: c:\Users\Sarad\Escritorio\Mates\Cursos\Curs 2025-2026 (3r + 4t)\TFG\TFG\.src\gtfs\data\stop_times_cleaned.txt


In [14]:
# Check that no trip_id from trip_ids_to_eliminate.txt appears in stop_times_cleaned.txt

def main():
    eliminate_path = os.path.join(BASE, "trip_ids_to_eliminate.txt")
    cleaned_path = os.path.join(BASE, "stop_times_cleaned.txt")
    check_missing_files([eliminate_path, cleaned_path])

    # Load trip_id to eliminate
    with open(eliminate_path, 'r', encoding='utf-8') as f:
        to_eliminate: Set[str] = {line.strip() for line in f if line.strip()}

    # Scan cleaned file for any remaining eliminated trip_id
    found: Set[str] = set()
    total_rows = 0
    for r in read_dict_rows(cleaned_path):
        total_rows += 1
        tid = r.get('trip_id', '').strip()
        if tid in to_eliminate:
            found.add(tid)

    print(f"Trip_id to eliminate: {len(to_eliminate)}")
    print(f"Total rows in stop_times_cleaned.txt: {total_rows}")

    if not found:
        print("All correct: no eliminated trip_id found in stop_times_cleaned.txt.")
    else:
        print(f"ERROR: {len(found)} eliminated trip_id still present in stop_times_cleaned.txt:")
        for tid in sorted(found):
            print(f"- {tid}")

main()

Trip_id to eliminate: 4111
Total rows in stop_times_cleaned.txt: 1060371
All correct: no eliminated trip_id found in stop_times_cleaned.txt.


#### Does stop_sequence increment by one?

Goal: check whether stop_sequence in stop_times.txt increments by one. To do this, we look at all 
trip_id from trips.txt. For each one, we go to stop_times.txt and look at the rows with that 
trip_id. We aggregate only the stop_sequence values per trip_id (without storing the full row) and check 
whether they increment by one. Optimized: we read stop_times once and aggregate by trip_id.
Parallel validation per trip with ThreadPoolExecutor to reduce total time.

In [15]:
def main():
    check_missing_files([STOP_TIMES_FILE, TRIPS_FILE])

    # trip_id of interest (filtered by prefix '1.')
    trip_ids = load_trip_ids(TRIPS_FILE)
    trip_ids = sorted(ids for ids in trip_ids if ids.startswith("1."))
    trip_id_set = set(trip_ids)

    print(f"Unique trip_id in trips (prefix '1.'): {len(trip_ids)}")

    # Build: trip_id -> set of stop_sequence (no duplicates), in a single pass over stop_times
    seq_by_trip: Dict[str, Set[int]] = defaultdict(set)
    for r in read_dict_rows(STOP_TIMES_FILE):
        tid = r.get('trip_id', '')
        if tid not in trip_id_set:
            continue
        seq_str = r.get('stop_sequence', '')
        try:
            seq = int(seq_str)
        except Exception:
            # Ignore non-numeric or empty values
            continue
        seq_by_trip[tid].add(seq)

    # Final map: trip_id -> sorted list of stop_sequence (no duplicates)
    dict_seq: Dict[str, List[int]] = {}
    for trip_id in trip_ids:
        dict_seq[trip_id] = sorted(seq_by_trip.get(trip_id, set()))

    # Parallel validation
    max_workers = min(8, (os.cpu_count() or 4))
    violations_total = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(check_trip, trip_id, dict_seq[trip_id]): trip_id for trip_id in trip_ids
        }
        for fut in as_completed(futures):
            msgs = fut.result()
            violations_total += len(msgs)
            for m in msgs:
                print(m)

    if violations_total == 0:
        print("Correct: all trips have stop_sequence that increments by one.")
    else:
        print(f"Total violations detected: {violations_total}")

main()


Unique trip_id in trips (prefix '1.'): 15088
Correct: all trips have stop_sequence that increments by one.


### Check that trips with trip_headsign Aeroport T1 don't only have 1 as direction_id

In [16]:
# Search for trip_id with trip_headsign == 'Aeroport T1' and direction_id == 0 in trips.txt

def main():
    check_missing_files([TRIPS_FILE])

    matches: List[str] = []
    total = 0
    for r in read_dict_rows(TRIPS_FILE):
        total += 1
        headsign = r.get('trip_headsign', '')
        direction = r.get('direction_id', '')
        if headsign == 'Aeroport T1' and direction == '0':
            tid = r.get('trip_id', '')
            if tid:
                matches.append(tid)

    print(f"Total rows in trips: {total}")
    if not matches:
        print("No trip found with trip_headsign='Aeroport T1' and direction_id=0.")
    else:
        print(f"Found {len(matches)} trip_id with trip_headsign='Aeroport T1' and direction_id=0:")
        for tid in sorted(matches):
            print(f"- {tid}")

main()

Total rows in trips: 48987
No trip found with trip_headsign='Aeroport T1' and direction_id=0.
